In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

LOD2_JSON = "data/lod2_meshes.json"
JSON_FILE = "data/lod3_meshes.json"
BLEND_FILE = "blender/lod3.blend"
GLB_FILE = "blender/lod3.glb"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
BPY_SCRIPT = "scripts/meshes_to_blender.py"
ORIGIN = (440000.0, 4473700.0, 640.0)   # Same local origin as LOD1 and LOD2

# Toledo facade line (from the Catastro, see NOTES.md): u = 0 at the corner with Calle de los Estudios.
TOLEDO_LINE = [(439988.6, 4473740.7), (439984.4, 4473757.7), (439978.7, 4473780.4), (439979.4, 4473792.5),
               (439980.8, 4473819.3), (439982.0, 4473827.4), (439984.2, 4473842.1), (439985.5, 4473850.4)]

# Courtyard of the cloister (Catastro: free area of parcel 0139424VK4703G), counterclockwise, so that
# "out" (left of the line) points into the courtyard. u = 0 at the NE corner.
CLOISTER_RING = [(440012.62, 4473806.32), (439995.59, 4473807.51), (439994.38, 4473790.2),
                 (440011.41, 4473789.01), (440012.62, 4473806.32)]

# North facade of the Colegiata on Calle de la Colegiata (Catastro), from west to east, so that "out"
# (left of the line) points to the street. u = 0 at the limit with parcel 01, u = 72.3 at parcel 03.
COLEGIATA_NORTH_LINE = [(440001.9, 4473857.8), (440040.3, 4473842.2), (440056.7, 4473833.8), (440067.7, 4473828.1)]

# Measured facades: table of elements, facade line, and sidewalk height (h = 0) in meters above sea level:
FACADES = [
    {"csv": "data/fachadas/toledo_instituto_u41-80.csv", "line": TOLEDO_LINE, "sidewalk": 644.4},
    {"csv": "data/fachadas/toledo_colegiata_u80-111.csv", "line": TOLEDO_LINE, "sidewalk": 644.4},
    {"csv": "data/fachadas/toledo_sur_u0-41.csv", "line": TOLEDO_LINE, "sidewalk": 644.4},   # All [ESTIMADO]
    {"csv": "data/fachadas/claustro.csv", "line": CLOISTER_RING, "sidewalk": 644.14},       # All [ESTIMADO]
    {"csv": "data/fachadas/colegiata_norte_u0-72.csv", "line": COLEGIATA_NORTH_LINE, "sidewalk": 645.2},   # All [ESTIMADO]
]
CYLINDER_SIDES = 16   # Column shafts are drawn as prisms with this many sides

# Finials (spire + iron cross) of the towers and the lantern, which the LOD2 grid cuts off.
# Center and tip height: highest building points of the cloud (spread of the tip points 0.14-0.34 m);
# base: top of the LOD2 roof there. The crosses are too thin for the cloud: size [ESTIMADO] from the 2022 photo.
FINIALS = [
    {"name": "remate_torre_norte", "x": 439989.0, "y": 4473845.9, "z_base": 686.4, "z_tip": 689.04, "radius": 0.5},
    {"name": "remate_torre_sur", "x": 439985.16, "y": 4473822.94, "z_base": 686.29, "z_tip": 689.12, "radius": 0.5},
    {"name": "remate_linterna", "x": 440028.34, "y": 4473826.81, "z_base": 699.47, "z_tip": 703.57, "radius": 0.8},
]
CROSS_HEIGHT = 1.5   # [ESTIMADO]
CROSS_ARM = 0.9      # [ESTIMADO]
CROSS_BAR = 0.12     # Thickness of the iron bars [ESTIMADO]

In [ ]:
# Celda 2
import csv
import json
import subprocess
import shapely
from shapely.geometry import LineString

In [ ]:
# Celda 3
# A point of the facade in local coordinates, from (u, n, h):
#   u = meters along the facade line, n = meters out of the wall (toward the street), h = meters above the sidewalk.
# The street is on the left of the line direction, so "out" is the left normal: direction (dx, dy) -> (-dy, dx).
def facade_point(line, u, n, h, direction, sidewalk):
    base = line.interpolate(u)
    dx, dy = direction
    x0, y0, z0 = ORIGIN
    return (base.x - dy * n - x0, base.y + dx * n - y0, sidewalk + h - z0)

# Unit direction of the line segment that contains u:
def direction_at(line, u):
    coords = list(line.coords)
    walked = 0.0
    for a, b in zip(coords[:-1], coords[1:]):
        length = shapely.Point(a).distance(shapely.Point(b))
        if walked + length >= u:
            return ((b[0] - a[0]) / length, (b[1] - a[1]) / length)
        walked += length
    return ((b[0] - a[0]) / length, (b[1] - a[1]) / length)

In [ ]:
# Celda 4
# Box of one facade element (8 vertices, 6 faces). Long elements (cornices) are cut at the bends of the line,
# so that each piece follows its own segment:
def element_boxes(line, row, sidewalk):
    u_low = min(float(row["u1"]), float(row["u2"]))
    u_high = max(float(row["u1"]), float(row["u2"]))
    h1, h2 = float(row["h1"]), float(row["h2"])
    n1, n2 = float(row["n1"]), float(row["n2"])

    # Places where the line bends, inside the element:
    cuts = [u_low]
    for corner in list(line.coords)[1:-1]:
        u_corner = line.project(shapely.Point(corner))
        if u_low < u_corner < u_high:
            cuts.append(u_corner)
    cuts.append(u_high)

    vertices = []
    faces = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        direction = direction_at(line, (a + b) / 2)
        start = len(vertices)
        for u in [a, b]:
            for n in [n1, n2]:
                for h in [h1, h2]:
                    vertices.append(facade_point(line, u, n, h, direction, sidewalk))
        # Vertex index = start + 4 * (u is b) + 2 * (n is n2) + (h is h2):
        faces += [
            (start + 0, start + 1, start + 3, start + 2),   # u = a
            (start + 4, start + 6, start + 7, start + 5),   # u = b
            (start + 0, start + 4, start + 5, start + 1),   # n = n1
            (start + 2, start + 3, start + 7, start + 6),   # n = n2
            (start + 0, start + 2, start + 6, start + 4),   # h = h1
            (start + 1, start + 5, start + 7, start + 3),   # h = h2
        ]
    return vertices, faces

In [ ]:
# Celda 5
# Column shaft: vertical cylinder whose diameter is u1 - u2, centered between n1 and n2 from the wall.
# Elements named "columna_<number>" are shafts; "columna_1_pedestal" or "columna_1_capitel" are boxes.
import math

def is_column(name):
    return name.startswith("columna_") and name.count("_") == 1

def column_mesh(line, row, sidewalk):
    u_center = (float(row["u1"]) + float(row["u2"])) / 2
    n_center = (float(row["n1"]) + float(row["n2"])) / 2
    radius = abs(float(row["u1"]) - float(row["u2"])) / 2
    h1, h2 = float(row["h1"]), float(row["h2"])
    direction = direction_at(line, u_center)
    vertices = []
    for k in range(CYLINDER_SIDES):
        angle = 2 * math.pi * k / CYLINDER_SIDES
        du = radius * math.cos(angle)
        dn = radius * math.sin(angle)
        vertices.append(facade_point(line, u_center + du, n_center + dn, h1, direction, sidewalk))
        vertices.append(facade_point(line, u_center + du, n_center + dn, h2, direction, sidewalk))
    faces = []
    for k in range(CYLINDER_SIDES):
        j = (k + 1) % CYLINDER_SIDES
        faces.append((2 * k, 2 * j, 2 * j + 1, 2 * k + 1))
    faces.append(tuple(2 * k for k in range(CYLINDER_SIDES)))          # Bottom
    faces.append(tuple(2 * k + 1 for k in range(CYLINDER_SIDES)))      # Top
    return vertices, faces

# Round arch: elements named "arco_..." are a rectangle from h1 to h2 (the springing line) with a half
# circle on top (radius = half the width), as a thin plate between n1 and n2.
ARCH_SEGMENTS = 12

def is_arch(name):
    return name.startswith("arco_")

def arch_mesh(line, row, sidewalk):
    u_left = max(float(row["u1"]), float(row["u2"]))
    u_right = min(float(row["u1"]), float(row["u2"]))
    u_center = (u_left + u_right) / 2
    radius = (u_left - u_right) / 2
    h1, h2 = float(row["h1"]), float(row["h2"])
    n1, n2 = float(row["n1"]), float(row["n2"])
    direction = direction_at(line, u_center)
    # Outline in (u, h): bottom right, then the half circle from right to left, then bottom left:
    outline = [(u_right, h1)]
    for k in range(ARCH_SEGMENTS + 1):
        angle = math.pi * k / ARCH_SEGMENTS
        outline.append((u_center - radius * math.cos(angle), h2 + radius * math.sin(angle)))
    outline.append((u_left, h1))
    vertices = []
    for u, h in outline:
        vertices.append(facade_point(line, u, n1, h, direction, sidewalk))
        vertices.append(facade_point(line, u, n2, h, direction, sidewalk))
    count = len(outline)
    faces = [tuple(2 * k for k in range(count)), tuple(2 * k + 1 for k in range(count))]   # Back and front
    for k in range(count):
        j = (k + 1) % count
        faces.append((2 * k, 2 * j, 2 * j + 1, 2 * k + 1))
    return vertices, faces

In [ ]:
# Celda 5b
# Shapes read on the photos (see NOTES.md, "Detalles"). The shape of each element comes from its name:
#   profiles: cornices, string courses, mouldings, entablatures, capitals of pilasters, balcony slabs (the section
#     of the box follows a profile, scaled to its height and projection);
#   "fronton": segmental pediment (arc band and tympanum); "columna_N": shaft with base and astragal;
#   "columna_N_capitel": bell and abacus; "oculo_": stone ring; "remate_bola_": finial (turned);
#   "mensulas_": a row of modillions; "almohadillado": banded rustication (one band per ashlar course);
#   "balcón_" / "balcon_reja_": iron railing (and slab); "reja_", "verja", "mirador_reja": flat grille;
#   silhouettes (escudos, cartouches, statues, eared frames): polygons drawn on the photos, in SILHOUETTES_CSV.
# Profiles: outline from the wall at the bottom (0, 0) outward and up to the wall at the top (0, 1), as fractions of
# the projection (n1..n2) and of the height (h1..h2). Shapes [ESTIMADO] read on the photos (sources 07, 08, 13).
import os
import re
import numpy as np
from shapely.geometry import Polygon
from shapely.geometry.polygon import orient

SILHOUETTES_CSV = "data/fachadas/siluetas.csv"
PROFILES = {
    # Fascia, cyma, corona with its drip and cymatium (Toledo, instituto: photo 01; Colegiata: photo 2022):
    "cornisa": [(0, 0), (0.15, 0), (0.15, 0.12), (0.25, 0.2), (0.35, 0.3), (0.45, 0.36), (0.45, 0.42), (0.95, 0.45),
                (1.0, 0.5), (1.0, 0.72), (0.92, 0.76), (0.92, 0.8), (0.97, 0.9), (1.0, 1.0), (0, 1.0)],
    "imposta": [(0, 0), (0.6, 0), (1.0, 0.25), (1.0, 0.75), (0.85, 1.0), (0, 1.0)],
    "moldura": [(0, 0), (0.4, 0), (0.4, 0.2), (0.7, 0.45), (1.0, 0.5), (1.0, 0.85), (0.9, 1.0), (0, 1.0)],
    "arquitrabe": [(0, 0), (0.6, 0), (0.6, 0.33), (0.8, 0.33), (0.8, 0.66), (1.0, 0.66), (1.0, 0.9), (0.9, 1.0), (0, 1.0)],
    "capitel": [(0, 0), (0.55, 0), (0.55, 0.12), (0.7, 0.3), (0.85, 0.55), (1.0, 0.62), (1.0, 1.0), (0, 1.0)],
    "losa": [(0, 0), (0.9, 0), (1.0, 0.3), (1.0, 0.7), (0.95, 1.0), (0, 1.0)],
    "basa": [(0, 0), (1.0, 0), (1.0, 0.7), (0.6, 1.0), (0, 1.0)],
    "mensula": [(0, 0), (0.35, 0), (0.7, 0.3), (1.0, 0.55), (1.0, 1.0), (0, 1.0)],
    "banda": [(0, 0), (0.6, 0), (1.0, 0.25), (1.0, 0.75), (0.6, 1.0), (0, 1.0)],
    "escalera": [(0, 0), (1.0, 0), (1.0, 0.33), (0.67, 0.33), (0.67, 0.67), (0.33, 0.67), (0.33, 1.0), (0, 1.0)],   # Three steps
}
PROFILE_RULES = [   # (pattern on the name, profile); the first that matches wins
    (r"capitel", "capitel"),
    (r"cornisa", "cornisa"),
    (r"imposta", "imposta"),
    (r"moldura|balaustrada_(base|barandal)", "moldura"),
    (r"entablamento|arquitrabe", "arquitrabe"),
    (r"losa", "losa"),
    (r"basa|pedestal", "basa"),
    (r"escalinata", "escalera"),
]
COURSE = 0.46            # Ashlar course of the instituto (autocorrelation on the rectified photo, as in 14_materiales)
JOINT = 0.05             # Open joint between two bands of the rustication [ESTIMADO]
MODILLION_SPACING = 1.0  # Colegiata: modillions about every meter (rectified photo 2022, 1 m grid)
MODILLION_WIDTH = 0.22   # [ESTIMADO]
BAR = 0.025              # Iron bars of railings and grilles [ESTIMADO]
BAR_SPACING = 0.13       # [ESTIMADO] from the 2022 photo (balconies) and photo 01 (basement windows)
LATHE_SIDES = 16

def merge(parts):
    vertices, faces = [], []
    for v, f in parts:
        start = len(vertices)
        vertices += v
        faces += [tuple(start + i for i in face) for face in f]
    return vertices, faces

def box(line, u1, u2, h1, h2, n1, n2, sidewalk):
    return element_boxes(line, {"u1": u1, "u2": u2, "h1": h1, "h2": h2, "n1": n1, "n2": n2}, sidewalk)

def bend_cuts(line, u_low, u_high):
    corners = [line.project(shapely.Point(c)) for c in list(line.coords)[1:-1]]
    return [u_low] + sorted(u for u in corners if u_low < u < u_high) + [u_high]

def profile_prism(line, u_low, u_high, outline, sidewalk):
    """Outline [(n, h)] (absolute) swept along the facade from u_low to u_high, cut at the bends of the line."""
    vertices, faces = [], []
    m = len(outline)
    cuts = bend_cuts(line, u_low, u_high)
    for a, b in zip(cuts[:-1], cuts[1:]):
        direction = direction_at(line, (a + b) / 2)
        start = len(vertices)
        for u in (a, b):
            vertices += [facade_point(line, u, n, h, direction, sidewalk) for n, h in outline]
        faces.append(tuple(start + k for k in range(m)))
        faces.append(tuple(start + m + k for k in reversed(range(m))))
        for k in range(m):
            j = (k + 1) % m
            faces.append((start + k, start + m + k, start + m + j, start + j))
    return vertices, faces

def scaled_profile(profile, row):
    h1, h2, n1, n2 = (float(row[k]) for k in ("h1", "h2", "n1", "n2"))
    return [(n1 + fn * (n2 - n1), h1 + fh * (h2 - h1)) for fn, fh in PROFILES[profile]]

def profile_for(name):
    return next((profile for pattern, profile in PROFILE_RULES if re.search(pattern, name)), None)

def polygon_prism(line, polygon, n1, n2, sidewalk):
    """Polygon (with holes) in (u, h), extruded from n1 to n2. Faces are triangulated, so holes are allowed."""
    direction = direction_at(line, polygon.centroid.x)
    vertices, faces = [], []
    def add(u, n, h):
        vertices.append(facade_point(line, u, n, h, direction, sidewalk))
        return len(vertices) - 1
    for poly in getattr(polygon, "geoms", [polygon]):
        poly = orient(poly, 1.0)
        for triangle in shapely.constrained_delaunay_triangles(poly).geoms:
            points = list(orient(triangle, 1.0).exterior.coords)[:3]
            faces.append(tuple(add(u, n1, h) for u, h in reversed(points)))
            faces.append(tuple(add(u, n2, h) for u, h in points))
        for ring_line in [poly.exterior, *poly.interiors]:
            c = list(ring_line.coords)[:-1]
            for (ua, ha), (ub, hb) in zip(c, c[1:] + c[:1]):
                faces.append((add(ua, n1, ha), add(ub, n1, hb), add(ub, n2, hb), add(ua, n2, ha)))
    return vertices, faces

def lathe(line, u_c, n_c, points_rh, sidewalk, sides=LATHE_SIDES):
    """Turned solid around a vertical axis at (u_c, n_c): points_rh = [(radius, h)] from bottom to top."""
    direction = direction_at(line, u_c)
    vertices, faces = [], []
    for r, h in points_rh:
        for k in range(sides):
            a = 2 * math.pi * k / sides
            vertices.append(facade_point(line, u_c + r * math.cos(a), n_c + r * math.sin(a), h, direction, sidewalk))
    for level in range(len(points_rh) - 1):
        for k in range(sides):
            j = (k + 1) % sides
            faces.append((level * sides + k, level * sides + j, (level + 1) * sides + j, (level + 1) * sides + k))
    faces.append(tuple(reversed(range(sides))))
    faces.append(tuple((len(points_rh) - 1) * sides + k for k in range(sides)))
    return vertices, faces

def segmental_pediment(u_a, u_b, h1, h2, band):
    """Segmental pediment over u_a..u_b, base h1, apex h2: (arc band + base cornice, tympanum) as polygons in (u, h)."""
    half, c, rise = abs(u_a - u_b) / 2, (u_a + u_b) / 2, h2 - h1
    radius = (half ** 2 + rise ** 2) / (2 * rise)
    hc = h2 - radius
    a0 = math.asin(min(half / radius, 1.0))
    segment = Polygon([(c + radius * math.sin(t), hc + radius * math.cos(t)) for t in np.linspace(-a0, a0, 24)])
    inner = shapely.Point(c, hc).buffer(radius - band, 64)
    base = shapely.box(c - half, h1, c + half, h1 + 0.6 * band)
    return segment.difference(inner).union(base), segment.intersection(inner)

def railing(line, u_low, u_high, h1, h2, n_front, sidewalk, sides=True):
    """Iron railing: vertical bars, rails at the bottom and the top, and (if sides) the two returns to the wall."""
    parts = []
    count = max(int((u_high - u_low) / BAR_SPACING), 1)
    for k in range(count + 1):
        u = u_low + k * (u_high - u_low) / count
        parts.append(box(line, u - BAR / 2, u + BAR / 2, h1, h2, n_front - BAR, n_front, sidewalk))
    for h in (h1, h2 - 2 * BAR):
        parts.append(box(line, u_low, u_high, h, h + 2 * BAR, n_front - 2 * BAR, n_front, sidewalk))
        if sides:
            for u in (u_low, u_high):
                parts.append(box(line, u - BAR, u + BAR, h, h + 2 * BAR, 0.0, n_front, sidewalk))
    if sides:
        for u in (u_low, u_high):
            for n in np.arange(BAR_SPACING, n_front - BAR, BAR_SPACING):
                parts.append(box(line, u - BAR / 2, u + BAR / 2, h1, h2, n - BAR / 2, n + BAR / 2, sidewalk))
    return merge(parts)

def grille(line, u_low, u_high, h1, h2, n1, n2, sidewalk):
    """Flat grille: vertical bars and a horizontal bar every meter (at least bottom, middle and top)."""
    parts = []
    count = max(int((u_high - u_low) / BAR_SPACING), 1)
    for k in range(count + 1):
        u = u_low + k * (u_high - u_low) / count
        parts.append(box(line, u - BAR / 2, u + BAR / 2, h1, h2, n1, n2, sidewalk))
    rows = max(int((h2 - h1) / 1.0), 2)
    for k in range(rows + 1):
        h = h1 + k * (h2 - h1 - 2 * BAR) / rows
        parts.append(box(line, u_low, u_high, h, h + 2 * BAR, n1, n2, sidewalk))
    return merge(parts)

def rustication(line, row, elements, sidewalk):
    """Bands of COURSE with open joints, interrupted where an opening, a frame or a portal is (same table)."""
    u_low, u_high = sorted((float(row["u1"]), float(row["u2"])))
    h1, h2, n1, n2 = (float(row[k]) for k in ("h1", "h2", "n1", "n2"))
    blockers = [r for r in elements if r is not row and (r["material"] == "vidrio" or r["elemento"].startswith("ventana"))]
    # A portal stops the bands at the outer edge of its pilaster bases, over the whole height of the bands:
    portals = {}
    for r in elements:
        if re.fullmatch(r"portada\d+_basa_(izq|der)", r["elemento"]):
            portals.setdefault(r["elemento"].split("_")[0], []).extend([float(r["u1"]), float(r["u2"])])
    for us in portals.values():
        blockers.append({"elemento": "portada", "u1": str(max(us)), "u2": str(min(us)), "h1": str(h1), "h2": str(h2)})
    parts = []
    h = h1
    while h + COURSE / 2 < h2:
        top = min(h + COURSE, h2)
        free = [(u_low, u_high)]
        for r in blockers:
            if float(r["h2"]) <= h or float(r["h1"]) >= top:
                continue
            a, b = sorted((float(r["u1"]), float(r["u2"])))
            free = [piece for lo, hi in free for piece in ((lo, min(hi, a)), (max(lo, b), hi)) if piece[1] - piece[0] > 0.2]
        outline = [(n1 + fn * (n2 - n1), h + JOINT / 2 + fh * (top - h - JOINT)) for fn, fh in PROFILES["banda"]]
        parts += [profile_prism(line, lo, hi, outline, sidewalk) for lo, hi in free]
        h = top
    return merge(parts)

def element_shapes(line, row, elements, sidewalk):
    """Meshes of one table row: list of (name suffix, group, (vertices, faces))."""
    name, group = row["elemento"], row["material"]
    u_low, u_high = sorted((float(row["u1"]), float(row["u2"])))
    h1, h2, n1, n2 = (float(row[k]) for k in ("h1", "h2", "n1", "n2"))
    u_c, n_c = (u_low + u_high) / 2, (n1 + n2) / 2
    if is_column(name):
        r = (u_high - u_low) / 2
        base = lathe(line, u_c, n_c, [(0, h1), (1.18 * r, h1), (1.18 * r, h1 + 0.12), (1.08 * r, h1 + 0.2), (1.12 * r, h1 + 0.28), (r, h1 + 0.36), (0, h1 + 0.36)], sidewalk)
        astragal = lathe(line, u_c, n_c, [(0, h2 - 0.2), (1.07 * r, h2 - 0.2), (1.07 * r, h2 - 0.08), (0, h2 - 0.08)], sidewalk)
        return [("", group, merge([column_mesh(line, row, sidewalk), base, astragal]))]
    if re.fullmatch(r"columna_\d+_capitel", name):
        r_top = (u_high - u_low) / 2 - 0.02
        bell = lathe(line, u_c, n_c, [(0, h1), (r_top - 0.2, h1), (r_top - 0.05, h1 + 0.5 * (h2 - h1)), (r_top, h1 + 0.65 * (h2 - h1)), (0, h1 + 0.65 * (h2 - h1))], sidewalk)
        abacus = box(line, u_low, u_high, h1 + 0.65 * (h2 - h1), h2, n1, n2, sidewalk)
        return [("", group, merge([bell, abacus]))]
    if is_arch(name):
        return [("", group, arch_mesh(line, row, sidewalk))]
    if name.startswith("oculo_"):
        r = (u_high - u_low) / 2
        ring_shape = shapely.Point(u_c, (h1 + h2) / 2).buffer(r, 32).difference(shapely.Point(u_c, (h1 + h2) / 2).buffer(r - 0.15, 32))
        return [("", group, polygon_prism(line, ring_shape, n1, n2, sidewalk))]
    if name.startswith("remate_bola"):
        r = min(u_high - u_low, n2 - n1) / 2
        dh = h2 - h1
        points = [(0, h1), (r, h1), (r, h1 + 0.25 * dh), (0.55 * r, h1 + 0.3 * dh), (0.45 * r, h1 + 0.38 * dh), (0.85 * r, h1 + 0.55 * dh),
                  (0.95 * r, h1 + 0.68 * dh), (0.8 * r, h1 + 0.82 * dh), (0.3 * r, h1 + 0.9 * dh), (0.2 * r, h1 + 0.97 * dh), (0, h2)]
        return [("", group, lathe(line, u_c, n_c, points, sidewalk))]
    if name.startswith("mensulas_"):
        count = max(int((u_high - u_low) / MODILLION_SPACING), 1)
        first = u_low + ((u_high - u_low) - (count - 1) * MODILLION_SPACING) / 2
        outline = scaled_profile("mensula", row)
        return [("", group, merge([profile_prism(line, first + k * MODILLION_SPACING - MODILLION_WIDTH / 2,
                                                 first + k * MODILLION_SPACING + MODILLION_WIDTH / 2, outline, sidewalk) for k in range(count)]))]
    if name.startswith("almohadillado"):
        return [("", group, rustication(line, row, elements, sidewalk))]
    if "fronton" in name or "frontón" in name:
        band, tympanum = segmental_pediment(u_low, u_high, h1, h2, band=min(0.35, 0.45 * (h2 - h1)))
        return [("", group, merge([polygon_prism(line, band, n1, n2, sidewalk), polygon_prism(line, tympanum, n1, n1 + 0.4 * (n2 - n1), sidewalk)]))]
    if name.startswith("balcón_"):   # Whole balcony of the Colegiata: slab and railing
        slab_top = h1 + 0.16
        slab = profile_prism(line, u_low - 0.05, u_high + 0.05, scaled_profile("losa", {"h1": h1, "h2": slab_top, "n1": 0.0, "n2": n2 + 0.05}), sidewalk)
        return [("_losa", "piedra", slab), ("_reja", "metal", railing(line, u_low, u_high, slab_top, h2, n2, sidewalk))]
    if name.startswith("balcon_reja"):
        return [("", "metal", railing(line, u_low, u_high, h1, h2, n2, sidewalk))]
    if name.startswith(("reja_", "verja")) or name == "mirador_reja":
        return [("", "metal", grille(line, u_low, u_high, h1, h2, n1, n2, sidewalk))]
    if name.startswith("balaustre_"):   # Turned baluster (photo 2022)
        r = (u_high - u_low) / 2
        dh = h2 - h1
        points = [(0, h1), (r, h1), (r, h1 + 0.1 * dh), (0.55 * r, h1 + 0.16 * dh), (0.95 * r, h1 + 0.38 * dh), (0.8 * r, h1 + 0.55 * dh),
                  (0.45 * r, h1 + 0.75 * dh), (0.45 * r, h1 + 0.84 * dh), (r, h1 + 0.9 * dh), (r, h2), (0, h2)]
        return [("", group, lathe(line, u_c, n_c, points, sidewalk, sides=8))]
    profile = profile_for(name)
    if profile:
        # Interrupted where an opening of the same table crosses it (a moulding does not run across a door):
        free = [(u_low, u_high)]
        for r in elements:
            if r["material"] != "vidrio":
                continue
            a, b = sorted((float(r["u1"]), float(r["u2"])))
            top = float(r["h2"]) + ((b - a) / 2 if is_arch(r["elemento"]) else 0)
            if top <= h1 or float(r["h1"]) >= h2:
                continue
            free = [piece for lo, hi in free for piece in ((lo, min(hi, a)), (max(lo, b), hi)) if piece[1] - piece[0] > 0.05]
        return [("", group, merge([profile_prism(line, lo, hi, scaled_profile(profile, row), sidewalk) for lo, hi in free]))]
    # Plain box; one mesh per segment of the facade line, so that each one is a flat plate (17_detalles cuts plates):
    vertices, faces = element_boxes(line, row, sidewalk)
    return [("" if k == 0 else f"_s{k}", group, (vertices[8 * k:8 * k + 8], [tuple(i - 8 * k for i in f) for f in faces[6 * k:6 * k + 6]]))
            for k in range(len(vertices) // 8)]

# Silhouettes: polygons in (u, h) meters of their facade, drawn on a photo (see the "origen" column):
with open(SILHOUETTES_CSV, encoding="utf-8") as f:
    silhouettes = list(csv.DictReader(f))
print(len(silhouettes), "silhouettes |", len(PROFILES), "profiles")

In [ ]:
# Celda 6
# Start from the LOD2 meshes and add the meshes of each facade element (Celda 5b) and of its silhouettes:
with open(LOD2_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]
print(len(meshes), "LOD2 meshes")

def facade_mesh(name, group, facade, origin, n2, vertices, faces):
    return {
        "name": name,
        "parcel": "",
        "group": group,
        "facade": facade["csv"],
        "thin": n2 <= 0.1,   # Plates drawn on the wall (openings, frames): replaced by the photo in 12_texturas
        "lod": 3,
        "height_m": 0.0,
        "height_method": origin,
        "vertices": vertices,
        "faces": faces,
    }

for facade in FACADES:
    line = LineString(facade["line"])
    with open(facade["csv"], encoding="utf-8") as f:
        elements = list(csv.DictReader(f))
    for row in elements:
        for suffix, group, (vertices, faces) in element_shapes(line, row, elements, facade["sidewalk"]):
            meshes.append(facade_mesh(row["elemento"] + suffix, group, facade, row["origen"], float(row["n2"]), vertices, faces))
    own = [s for s in silhouettes if s["fachada"] == os.path.basename(facade["csv"])]
    for s in own:
        vertices, faces = polygon_prism(line, shapely.from_wkt(s["wkt"]), float(s["n1"]), float(s["n2"]), facade["sidewalk"])
        meshes.append(facade_mesh(s["elemento"], s["material"], facade, s["origen"], float(s["n2"]), vertices, faces))
    print(len(elements), "elements and", len(own), "silhouettes from", facade["csv"])
print(len(meshes), "meshes in total")

In [ ]:
# Celda 7
# Finials: an 8-sided spire from z_base (sunk 0.5 m into the roof) to z_tip, and a cross on top.
# The cross arm runs north-south, along the Toledo facade, so that it faces the street.
def spire_mesh(x, y, z_bottom, z_top, radius, sides=8):
    x0, y0, z0 = ORIGIN
    vertices = []
    for k in range(sides):
        angle = 2 * math.pi * k / sides
        vertices.append((x + radius * math.cos(angle) - x0, y + radius * math.sin(angle) - y0, z_bottom - z0))
    vertices.append((x - x0, y - y0, z_top - z0))   # Apex
    apex = sides
    faces = [(k, (k + 1) % sides, apex) for k in range(sides)]
    faces.append(tuple(reversed(range(sides))))      # Bottom
    return vertices, faces

def box_mesh(xmin, xmax, ymin, ymax, zmin, zmax):
    x0, y0, z0 = ORIGIN
    vertices = []
    for x in [xmin, xmax]:
        for y in [ymin, ymax]:
            for z in [zmin, zmax]:
                vertices.append((x - x0, y - y0, z - z0))
    # Vertex index = 4 * (x is max) + 2 * (y is max) + (z is max):
    faces = [(0, 1, 3, 2), (4, 6, 7, 5), (0, 4, 5, 1), (2, 3, 7, 6), (0, 2, 6, 4), (1, 5, 7, 3)]
    return vertices, faces

half = CROSS_BAR / 2
for finial in FINIALS:
    x, y, tip = finial["x"], finial["y"], finial["z_tip"]
    vertices, faces = spire_mesh(x, y, finial["z_base"] - 0.5, tip, finial["radius"])
    meshes.append({"name": finial["name"] + "_aguja", "parcel": "0139402VK4703G", "group": "pizarra", "lod": 3,
                   "height_m": 0.0, "height_method": "punta medida en la nube", "vertices": vertices, "faces": faces})
    upright = box_mesh(x - half, x + half, y - half, y + half, tip - 0.2, tip + CROSS_HEIGHT)
    arm_z = tip + CROSS_HEIGHT * 0.65
    arm = box_mesh(x - half, x + half, y - CROSS_ARM / 2, y + CROSS_ARM / 2, arm_z - half, arm_z + half)
    for part_name, (vertices, faces) in [("_cruz_mastil", upright), ("_cruz_brazo", arm)]:
        meshes.append({"name": finial["name"] + part_name, "parcel": "0139402VK4703G", "group": "metal", "lod": 3,
                       "height_m": 0.0, "height_method": "[ESTIMADO] tamaño de la cruz", "vertices": vertices, "faces": faces})
print(len(FINIALS), "finials |", len(meshes), "meshes in total")

In [ ]:
# Celda 8
# Save and run Blender:
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": meshes}, f)

command = [BLENDER, "-b", "--factory-startup", "--python", BPY_SCRIPT, "--", JSON_FILE, BLEND_FILE, GLB_FILE]
result = subprocess.run(command, capture_output=True, text=True)
for line_text in result.stdout.splitlines():
    if "Objects" in line_text or "Error" in line_text or "Saved" in line_text:
        print(line_text)
print("Return code:", result.returncode)
if result.returncode != 0:
    print(result.stderr[-2000:])